In [12]:
#import libraries

from bs4 import BeautifulSoup
import requests
import smtplib
import time
import datetime
import re
import csv
import os

In [13]:
URL = 'https://www.amazon.com/Pok%C3%A9mon-Pokopia-Expansion-Digital-Nintendo/dp/B0H4NNTW5T/ref=sr_1_2?...'

# https://httpbin.org/get
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Referer": "https://www.amazon.com/"
}

page = requests.get(URL, headers=headers)

soup = BeautifulSoup(page.content, "html.parser")

title_tag = soup.find("span", id="productTitle")
print("Title tag:", title_tag)

if title_tag:
    print("Title text:", title_tag.get_text().strip())
else:
    print("Title not found.")
    print(page.text[:800])

Title tag: <span class="a-size-large product-title-word-break" id="productTitle">        Pokémon Pokopia Bundle (Game + Expansion Pass) | Digital | Nintendo Switch 2       </span>
Title text: Pokémon Pokopia Bundle (Game + Expansion Pass) | Digital | Nintendo Switch 2


In [14]:
title_tag = soup.find("span", id="productTitle")

if title_tag:
    full_title = title_tag.get_text().strip()
    print(full_title)
else:
    print("Title not found.")

Pokémon Pokopia Bundle (Game + Expansion Pass) | Digital | Nintendo Switch 2


In [15]:
# Titel
title_tag = soup.find("span", id="productTitle")
title = title_tag.get_text().strip() if title_tag else "Title not found"

# Price
price_tag = soup.find("span", {"data-a-price": True})  # eller accessibility label

if not price_tag:
    price_tag = soup.find("span", class_="a-price")

if price_tag:
    # Tries to get price from aria-hidden or text
    price = price_tag.get_text().strip()
    
    # Alternative: searches for EUR + number
    import re
    match = re.search(r'(\d+[.,]\d{2})', price_tag.get_text())
    if match:
        price_str = match.group(1).replace(',', '.')
        price = float(price_str)
    else:
        price = price
else:
    price = "Price not found"

print("Title:", title)
print("Price:", price)

Title: Pokémon Pokopia Bundle (Game + Expansion Pass) | Digital | Nintendo Switch 2
Price: 92.22


In [16]:
# Price + Currency
price_tag = soup.find("span", {"data-a-price": True})
if not price_tag:
    price_tag = soup.find("span", class_="a-price")

price_symbol = soup.find("span", class_="a-price-symbol")

if price_tag:
    # Get Price
    import re
    match = re.search(r'(\d+[.,]\d{2})', price_tag.get_text() or "")
    if match:
        price_str = match.group(1).replace(',', '.')
        price = float(price_str)
    else:
        price = None
else:
    price = None

# Get Currency
currency = price_symbol.get_text().strip() if price_symbol else "EUR"

currency_price = f"{currency} {price:.2f}" if price is not None else "Price not found"

print("Title:", title)
print("Currency Price:", currency_price)   # "EUR 92.04"
print("Price as float:", price)    # "92.04"

Title: Pokémon Pokopia Bundle (Game + Expansion Pass) | Digital | Nintendo Switch 2
Currency Price: EUR 92.22
Price as float: 92.22


In [17]:
# Ratings and Reviews
rating_tag = soup.find("span", class_="a-size-small a-color-base")
reviews_tag = soup.find("span", id="acrCustomerReviewText")

# Rating (eg. 5.0)
if rating_tag:
    rating_text = rating_tag.get_text().strip()
    # Exctract solely the number
    match = re.search(r'(\d+\.?\d*)', rating_text)
    rating = float(match.group(1)) if match else None
else:
    rating = None

# X amount of reviews (eg. 2)
if reviews_tag:
    reviews_text = reviews_tag.get_text().strip()
    match = re.search(r'(\d+)', reviews_text)
    reviews = int(match.group(1)) if match else 0
else:
    reviews = 0

print("Title:", title)
print("currency:", currency)
print("Currency Price:", currency_price)
print("Price as float:", price)
print("Rating:", rating)           # eg. 5.0
print("Number of Reviews:", reviews)   # eg. 2

Title: Pokémon Pokopia Bundle (Game + Expansion Pass) | Digital | Nintendo Switch 2
currency: EUR
Currency Price: EUR 92.22
Price as float: 92.22
Rating: 5.0
Number of Reviews: 2


In [19]:
# Prepare for csv export

# Ratings and Reviews
rating_tag = soup.find("span", class_="a-size-small a-color-base")
reviews_tag = soup.find("span", id="acrCustomerReviewText")

# Rating (eg. 5.0)
if rating_tag:
    rating_text = rating_tag.get_text().strip()
    # Exctract solely the number
    match = re.search(r'(\d+\.?\d*)', rating_text)
    rating = float(match.group(1)) if match else None
else:
    rating = None

# X amount of reviews (eg. 2)
if reviews_tag:
    reviews_text = reviews_tag.get_text().strip()
    match = re.search(r'(\d+)', reviews_text)
    reviews = int(match.group(1)) if match else 0
else:
    reviews = 0

print(title)
print(currency)
print(currency_price)
print(price)
print(rating)
print(reviews)

Pokémon Pokopia Bundle (Game + Expansion Pass) | Digital | Nintendo Switch 2
EUR
EUR 92.22
92.22
5.0
2


In [20]:
# Define path for export of csv
downloads_path = r"C:\Users\wijks\Downloads"
filename = "AmazonWebScraperDataset.csv"
full_path = os.path.join(downloads_path, filename)

header = ['Title', 'Currency', 'Price with Currency', 'Price', 'Rating', 'Number of reviews']
data = [title, currency,currency_price, price, rating, reviews]

with open(full_path, 'w', newline='', encoding='UTF8') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerow(data)

print(f"CSV-file was downloaded to:")
print(full_path)

CSV-file was downloaded to:
C:\Users\wijks\Downloads\AmazonWebScraperDataset.csv
